Imports & constants

In [1]:
# Cell 1: imports and constants
import os
import random
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, average_precision_score)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = Path("Results/dhi_hybrid")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)

Device: cuda


Quick dataset inspection 

In [4]:
df = pd.read_csv('/home/usman/Desktop/RetinalDL_Single/TextDL/Datasets/dhi.csv')
print("Shape:", df.shape)
print(df.columns.tolist())
display(df.head())
display(df.describe(include='all').T)

Shape: (253680, 22)
['Diabetes_binary', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


,count,mean,std,min,25%,50%,75%,max
Diabetes_binary,253680.0,0.139333,0.346294,0.0,0.0,0.0,0.0,1.0
HighBP,253680.0,0.429001,0.494934,0.0,0.0,0.0,1.0,1.0
HighChol,253680.0,0.424121,0.494210,0.0,0.0,0.0,1.0,1.0
CholCheck,253680.0,0.962670,0.189571,0.0,1.0,1.0,1.0,1.0
BMI,253680.0,28.382364,6.608694,12.0,24.0,27.0,31.0,98.0
Smoker,253680.0,0.443169,0.496761,0.0,0.0,0.0,1.0,1.0
Stroke,253680.0,0.040571,0.197294,0.0,0.0,0.0,0.0,1.0
HeartDiseaseorAttack,253680.0,0.094186,0.292087,0.0,0.0,0.0,0.0,1.0
PhysActivity,253680.0,0.756544,0.429169,0.0,1.0,1.0,1.0,1.0
Fruits,253680.0,0.634256,0.481639,0.0,0.0,1.0,1.0,1.0


Preprocessing helper: impute, encode, scale, build sequences

In [5]:
# Cell 3: preprocessing function
from sklearn.impute import SimpleImputer

def preprocess_and_build_sequences(df, target_col='Diabetes_binary', scale=True, seq_mode='feature-as-timestep'):
    """
    Returns X_seq, y arrays ready for RNN input.
    seq_mode:
      - 'feature-as-timestep' -> treat each feature as timestep, shape -> (N, seq_len, 1)
    """
    df = df.copy()
    # drop duplicates if any
    df.drop_duplicates(inplace=True)

    # Separate target
    assert target_col in df.columns, f"{target_col} not in dataframe"
    y = df[target_col].astype(int).values
    X = df.drop(columns=[target_col]).reset_index(drop=True)

    # Simple numeric imputation for all columns
    num_cols = X.columns.tolist()
    imputer = SimpleImputer(strategy='median')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=num_cols)

    # Optionally: one-hot encode any nominal columns if present. For this dataset columns appear numeric already.
    # But if some columns are truly categorical encoded as numbers, you can one-hot them here.

    # Feature scaling
    scaler = None
    if scale:
        scaler = StandardScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=num_cols)
    else:
        X_scaled = X_imputed

    # Build sequences:
    if seq_mode == 'feature-as-timestep':
        # Each sample becomes sequence of shape (seq_len = n_features, input_size = 1)
        X_seq = X_scaled.values.astype(np.float32)
        N, F = X_seq.shape
        X_seq = X_seq.reshape(N, F, 1)   # (N, seq_len, input_size=1)
    else:
        raise ValueError("Unknown seq_mode")

    return X_seq, y, scaler, imputer, num_cols

# run preprocessing
X_seq, y, scaler, imputer, feature_names = preprocess_and_build_sequences(df, target_col='Diabetes_binary')
print("X_seq shape (N, seq_len, input_size):", X_seq.shape)
print("y shape:", y.shape)


X_seq shape (N, seq_len, input_size): (229474, 21, 1)
y shape: (229474,)


Dataset wrapper (PyTorch)

In [6]:
# Cell 4: PyTorch Dataset
class TabularSequenceDataset(Dataset):
    def __init__(self, X_seq, y):
        self.X = torch.tensor(X_seq, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

Model definition: Dense -> BN -> Dropout -> BiLSTM + BiGRU -> concat -> Dense

In [7]:
# Cell 5: Hybrid model
class HybridBiRNN(nn.Module):
    def __init__(self, seq_len, input_size=1,
                 lstm_hidden=64, gru_hidden=64,
                 dense1=128, dense2=64, dropout=0.2):
        super().__init__()
        self.seq_len = seq_len
        self.input_size = input_size

        # initial dense block applied per-sample (we'll flatten over features first)
        # But better: apply TimeDistributed Dense -> emulate using a linear that maps input_size -> dense_intermediate applied at each timestep
        self.timestep_fc = nn.Linear(input_size, dense1)
        self.timestep_bn = nn.BatchNorm1d(seq_len)  # normalize across features across batch/time
        self.timestep_drop = nn.Dropout(dropout)

        # BiLSTM branch
        self.lstm = nn.LSTM(input_size=dense1, hidden_size=lstm_hidden,
                            num_layers=1, batch_first=True, bidirectional=True)

        # BiGRU branch
        self.gru = nn.GRU(input_size=dense1, hidden_size=gru_hidden,
                          num_layers=1, batch_first=True, bidirectional=True)

        # After concatenation (lstm_out pooled + gru_out pooled)
        combined_size = (lstm_hidden * 2) + (gru_hidden * 2)

        self.fc_comb1 = nn.Linear(combined_size, dense2)
        self.bn_comb1 = nn.BatchNorm1d(dense2)
        self.drop_comb1 = nn.Dropout(dropout)
        self.fc_out = nn.Linear(dense2, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        b, s, i = x.shape
        # apply linear per timestep
        # reshape to (batch*seq_len, input_size) -> apply -> reshape back
        x_flat = x.view(b * s, i)
        x_flat = F.relu(self.timestep_fc(x_flat))  # (b*s, dense1)
        x_flat = x_flat.view(b, s, -1)  # (b, seq_len, dense1)

        # BiLSTM branch
        lstm_out, _ = self.lstm(x_flat)  # (b, seq_len, 2*lstm_hidden)
        # pool across time (mean pooling)
        lstm_pooled = torch.mean(lstm_out, dim=1)  # (b, 2*lstm_hidden)

        # BiGRU branch
        gru_out, _ = self.gru(x_flat)  # (b, seq_len, 2*gru_hidden)
        gru_pooled = torch.mean(gru_out, dim=1)  # (b, 2*gru_hidden)

        # concat
        x_cat = torch.cat([lstm_pooled, gru_pooled], dim=1)
        x = F.relu(self.bn_comb1(self.fc_comb1(x_cat)))
        x = self.drop_comb1(x)
        logits = self.fc_out(x)
        return logits


Training helpers (train_epoch, eval_epoch, early stopping)

In [8]:
# Cell 6: training utilities
def compute_class_weights(y):
    # returns weight for positive class for BCEWithLogitsLoss pos_weight
    classes, counts = np.unique(y, return_counts=True)
    # pos_weight = N_neg / N_pos
    if 1 in classes:
        pos = counts[classes.tolist().index(1)]
        neg = counts[classes.tolist().index(0)]
        pos_weight = torch.tensor(neg / (pos + 1e-8), dtype=torch.float32, device=DEVICE)
    else:
        pos_weight = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
    return pos_weight

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []
    for Xb, yb in loader:
        Xb = Xb.to(DEVICE); yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * Xb.size(0)

        probs = torch.sigmoid(logits).detach().cpu().numpy().ravel()
        preds = (probs >= 0.5).astype(int)
        y_true.extend(yb.cpu().numpy().ravel().astype(int))
        y_pred.extend(preds)
    avg_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return avg_loss, acc, f1

def eval_model(model, loader, criterion, threshold=0.5):
    model.eval()
    running_loss = 0.0
    y_true, y_probs = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE); yb = yb.to(DEVICE)
            logits = model(Xb)
            loss = criterion(logits, yb)
            running_loss += loss.item() * Xb.size(0)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            y_probs.extend(probs)
            y_true.extend(yb.cpu().numpy().ravel().astype(int))
    avg_loss = running_loss / len(loader.dataset)
    y_pred = (np.array(y_probs) >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    return avg_loss, acc, f1, np.array(y_true), np.array(y_probs)


5-Fold CV training loop (full)

In [10]:
# Cell 7: 5-fold CV training
from sklearn.model_selection import StratifiedKFold

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

# hyperparams
BATCH_SIZE = 256
EPOCHS = 120
PATIENCE = 15
LR = 1e-3
DROP = 0.3

seq_len = X_seq.shape[1]
input_size = X_seq.shape[2]

fold_results = []
models_paths = []

# compute pos_weight for BCE (based on whole dataset)
pos_weight = compute_class_weights(y)
print("pos_weight:", pos_weight.item())

for fold, (train_idx, val_idx) in enumerate(skf.split(X_seq, y), 1):
    print(f"\n=== Fold {fold}/{n_splits} ===")
    X_train, y_train = X_seq[train_idx], y[train_idx]
    X_val, y_val = X_seq[val_idx], y[val_idx]

    # DataLoaders
    train_ds = TabularSequenceDataset(X_train, y_train)
    val_ds = TabularSequenceDataset(X_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    # instantiate model
    model = HybridBiRNN(seq_len=seq_len, input_size=input_size,
                        lstm_hidden=64, gru_hidden=64, dense1=128, dense2=64, dropout=DROP)
    model.to(DEVICE)

    # optimizer, loss, scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=6, verbose=True)

    best_val_loss = float('inf')
    epochs_no_improve = 0

    # storage per epoch
    history = defaultdict(list)

    for epoch in range(1, EPOCHS+1):
        train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_f1, _, _ = eval_model(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)

        print(f"Fold {fold} | Epoch {epoch:03d} | Train loss {train_loss:.4f} acc {train_acc:.4f} f1 {train_f1:.4f} | "
              f"Val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f}")

        scheduler.step(val_loss)

        # early stopping
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            epochs_no_improve = 0
            # save model
            model_path = RESULTS_DIR / f"hybrid_fold{fold}_best.pth"
            torch.save(model.state_dict(), model_path)
            best_model_path = model_path
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"Early stopping fold {fold} at epoch {epoch}")
                break

    # load best model for evaluation on validation fold
    model.load_state_dict(torch.load(best_model_path))
    val_loss, val_acc, val_f1, val_true, val_probs = eval_model(model, val_loader, criterion)
    fold_results.append({
        'fold': fold,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'val_f1': val_f1,
        'val_true': val_true,
        'val_probs': val_probs,
        'history': history,
        'model_path': str(best_model_path)
    })
    models_paths.append(str(best_model_path))

    print(f"Fold {fold} result -> Val acc: {val_acc:.4f}, Val f1: {val_f1:.4f}, Val loss: {val_loss:.4f}")

# aggregate
accs = [r['val_acc'] for r in fold_results]
f1s = [r['val_f1'] for r in fold_results]
print("\n=== Cross-validated results ===")
print(f"Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"F1:       {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

# Save summary
with open(RESULTS_DIR / "cv_summary.json", "w") as f:
    json.dump({'accs': accs, 'f1s': f1s, 'models': models_paths}, f)


pos_weight: 5.5382795333862305

=== Fold 1/5 ===


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

Evaluate on hold-out test set

In [ ]:
# Cell 8: optional hold-out test set evaluation
# If you want a final hold-out test evaluation, create a train/test split before cross-val.
# Here we provide a function to evaluate saved models on an external X_test,y_test.

def evaluate_on_test(model_path, X_test_seq, y_test, threshold=0.5):
    model = HybridBiRNN(seq_len=X_test_seq.shape[1], input_size=X_test_seq.shape[2],
                        lstm_hidden=64, gru_hidden=64, dense1=128, dense2=64, dropout=DROP)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    ds = TabularSequenceDataset(X_test_seq, y_test)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    criterion = nn.BCEWithLogitsLoss()  # for reporting only
    loss, acc, f1, y_true, y_probs = eval_model(model, loader, criterion, threshold=threshold)
    return loss, acc, f1, y_true, y_probs

# Example:
# X_train_full, X_test_seq, y_train_full, y_test = train_test_split(X_seq, y, test_size=0.1, random_state=SEED, stratify=y)
# test_loss, test_acc, test_f1, test_true, test_probs = evaluate_on_test(models_paths[0], X_test_seq, y_test)
# print("Test acc:", test_acc, "Test f1:", test_f1, "Test loss:", test_loss)


Plots per fold and overall curves (loss/acc/F1, ROC, PR, confusion)

In [ ]:
# Cell 9: plotting functions
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def plot_fold_history(history, fold, savepath=None):
    epochs = range(1, len(history['train_loss'])+1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1)
    plt.plot(epochs, history['train_loss'], label='train loss')
    plt.plot(epochs, history['val_loss'], label='val loss')
    plt.title(f'Fold {fold} Loss'); plt.legend()
    plt.subplot(1,3,2)
    plt.plot(epochs, history['train_acc'], label='train acc')
    plt.plot(epochs, history['val_acc'], label='val acc')
    plt.title(f'Fold {fold} Acc'); plt.legend()
    plt.subplot(1,3,3)
    plt.plot(epochs, history['train_f1'], label='train f1')
    plt.plot(epochs, history['val_f1'], label='val f1')
    plt.title(f'Fold {fold} F1'); plt.legend()
    if savepath:
        plt.savefig(savepath, bbox_inches='tight', dpi=200)
    plt.show()

def plot_roc_pr(y_true, y_probs, title_prefix='', savepath=None):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)
    precision, recall, _ = precision_recall_curve(y_true, y_probs)
    ap = average_precision_score(y_true, y_probs)

    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
    plt.plot([0,1],[0,1],'--')
    plt.title(f'{title_prefix} ROC'); plt.legend()
    plt.subplot(1,2,2)
    plt.plot(recall, precision, label=f'AP = {ap:.3f}')
    plt.title(f'{title_prefix} Precision-Recall'); plt.legend()
    if savepath:
        plt.savefig(savepath, bbox_inches='tight', dpi=200)
    plt.show()

def plot_confusion(y_true, y_pred, title='', savepath=None):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot()
    plt.title(title)
    if savepath:
        plt.savefig(savepath, bbox_inches='tight', dpi=200)
    plt.show()

# Example display for first fold:
for r in fold_results[:1]:
    plot_fold_history(r['history'], r['fold'], savepath=RESULTS_DIR / f"fold{r['fold']}_history.png")
    plot_roc_pr(r['val_true'], r['val_probs'], title_prefix=f"Fold {r['fold']}", savepath=RESULTS_DIR / f"fold{r['fold']}_rocpr.png")
    preds = (r['val_probs'] >= 0.5).astype(int)
    plot_confusion(r['val_true'], preds, title=f"Fold {r['fold']} Confusion", savepath=RESULTS_DIR / f"fold{r['fold']}_cm.png")

# Overall ROC/PR by concatenating folds:
all_true = np.concatenate([r['val_true'] for r in fold_results])
all_probs = np.concatenate([r['val_probs'] for r in fold_results])
plot_roc_pr(all_true, all_probs, title_prefix='All folds', savepath=RESULTS_DIR / "allfolds_rocpr.png")
plot_confusion(all_true, (all_probs>=0.5).astype(int), title='All folds Confusion', savepath=RESULTS_DIR / "allfolds_cm.png")


Statistical testing: t-test vs ANOVA (which to use + code)

In [ ]:
# Cell 10: statistics - which test and code
"""
Which test to use?
- If you compare exactly TWO models (e.g., your original BiLSTM baseline vs this hybrid),
  use a paired t-test on their per-fold accuracies (paired because same folds used).
  Assumptions: differences approximately normally distributed. If not, use Wilcoxon signed-rank test.

- If you compare MORE THAN TWO models, use repeated-measures ANOVA (or Friedman test for non-parametric).
  Implementation below shows paired t-test and a fallback Wilcoxon and also repeated measures ANOVA using statsmodels if needed.
"""

# We'll show paired t-test comparing two arrays of per-fold accuracies:
from scipy import stats

# Example: If you have baseline_accs (list of 5) and hybrid_accs (list of 5)
# For demonstration, take baseline as previous BiLSTM values you reported or create a placeholder:
# baseline_accs = [0.84, 0.85, 0.85, 0.86, 0.85]  # replace with real numbers
# hybrid_accs = accs  # from the CV above

def paired_ttest(baseline_accs, new_accs):
    baseline = np.array(baseline_accs)
    new = np.array(new_accs)
    diff = new - baseline
    print("Baseline mean:", baseline.mean(), "New mean:", new.mean(), "Diff mean:", diff.mean())
    # Normality test on differences
    shapiro_p = stats.shapiro(diff).pvalue
    print("Shapiro-Wilk p for differences:", shapiro_p)
    if shapiro_p > 0.05:
        tstat, pval = stats.ttest_rel(new, baseline)
        print(f"Paired t-test t={tstat:.4f}, p={pval:.4f}")
    else:
        # fallback nonparametric
        stat, pval = stats.wilcoxon(new, baseline)
        print(f"Wilcoxon signed-rank stat={stat:.4f}, p={pval:.4f}")

# Save your baseline per-fold accuracies and call paired_ttest(baseline_accs, accs)
# Example usage (uncomment and replace baseline_accs with your numbers):
# baseline_accs = [0.85,0.85,0.85,0.85,0.85]  # put real baseline fold accuracies
# paired_ttest(baseline_accs, accs)

# If you have >2 models, consider repeated measures ANOVA:
# Example stub for repeated measures ANOVA (requires statsmodels)
"""
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.anova import AnovaRM

# Construct DataFrame in wide format with columns: subject (fold), modelA, modelB, modelC
# Then melt to long and run AnovaRM or use pingouin:
"""


Save results and model artifacts

In [ ]:
# Cell 11: save final artifacts
import joblib
# Save preprocessing objects
joblib.dump(scaler, RESULTS_DIR / "scaler.joblib")
joblib.dump(imputer, RESULTS_DIR / "imputer.joblib")
# Save fold results summary
with open(RESULTS_DIR / "fold_results_summary.json", "w") as f:
    json.dump({
        'accs': accs, 'f1s': f1s, 'models': models_paths
    }, f, indent=2)
print("Saved artifacts to", RESULTS_DIR)